# RAG 답변 유사도 평가

`data/chroma_db`의 `pet_care` 컬렉션과 validation CSV를 이용해, 질문을 넣었을 때 생성된 RAG 답변이 정답 답변(`qa.output`)과 얼마나 유사한지 평가합니다.

메인 지표는 답변 임베딩 cosine similarity입니다.

- `generated_answer_similarity`: RAG 생성 답변과 정답 답변의 유사도
- `retrieved_answer_similarity`: 검색된 근거 답변 중 정답 답변과 가장 유사한 값
- `precision@k`, `recall@k`, `mrr@k`: 검색 품질 참고용 지표

주의: 정답 metadata를 검색 필터로 넣지 않습니다. 실제 질문 입력 상황에 가까운 평가를 위해 질문만으로 검색합니다.

In [11]:
from pathlib import Path
from collections import Counter
import os
import tomllib

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

CHROMA_DIR = PROJECT_DIR / "data" / "chroma_db"
VAL_PATH = PROJECT_DIR / "data" / "df_val.csv"
# 기타 제외 CSV로 평가하려면 아래 줄을 사용하세요.
# VAL_PATH = PROJECT_DIR / "data" / "df_val_without_etc.csv"

COLLECTION_NAME = "pet_care"
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"
GENERATION_MODEL_NAME = "gpt-5.6-luna"

ANSWER_TOP_K = 3
K_VALUES = [1, 3, 5]
MAX_K = max(max(K_VALUES), ANSWER_TOP_K)
METADATA_KEYS = ["meta.lifeCycle", "meta.department", "meta.disease"]

# 비용과 시간을 줄이기 위해 기본은 50개만 생성 평가합니다. 전체 평가는 None으로 바꾸세요.
EVAL_LIMIT = 50
RANDOM_SAMPLE = False
RANDOM_SEED = 42

load_dotenv(PROJECT_DIR / ".env")

print("project:", PROJECT_DIR)
print("validation:", VAL_PATH)
print("chroma:", CHROMA_DIR)

project: c:\Users\rhksa\Desktop\mle-01-p1-team2
validation: c:\Users\rhksa\Desktop\mle-01-p1-team2\data\df_val.csv
chroma: c:\Users\rhksa\Desktop\mle-01-p1-team2\data\chroma_db


In [12]:
def get_openai_api_key() -> str | None:
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        return api_key

    secrets_path = PROJECT_DIR / ".streamlit" / "secrets.toml"
    if not secrets_path.exists():
        return None

    with secrets_path.open("rb") as file:
        secrets = tomllib.load(file)
    return secrets.get("OPENAI_API_KEY")


OPENAI_API_KEY = get_openai_api_key()
print("openai key loaded:", bool(OPENAI_API_KEY))

openai key loaded: True


In [13]:
val_df = pd.read_csv(VAL_PATH, encoding="utf-8-sig")

required_columns = ["qa.input", "qa.output", *METADATA_KEYS]
missing_columns = [column for column in required_columns if column not in val_df.columns]
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")

val_df = val_df.dropna(subset=required_columns).reset_index(drop=True)

if EVAL_LIMIT is not None:
    if RANDOM_SAMPLE:
        eval_df = val_df.sample(n=min(EVAL_LIMIT, len(val_df)), random_state=RANDOM_SEED).reset_index(drop=True)
    else:
        eval_df = val_df.head(EVAL_LIMIT).copy().reset_index(drop=True)
else:
    eval_df = val_df.copy().reset_index(drop=True)

print("validation rows:", len(val_df))
print("evaluation rows:", len(eval_df))
display(eval_df.head(3))

validation rows: 560
evaluation rows: 50


,Unnamed: 0,meta.lifeCycle,meta.department,meta.disease,qa.input,qa.output
0,5,노령견,내과,심장사상충,저희 집에서 기르고 있는 강아지는 11세의 수컷입니다. 얼마 전 동물병원에 방문했을...,답변드립니다. 1. 청진을 통해 심장에 잡음이 발생하는 것을 확인할 수 있었습니다....
1,10,성견,내과,방광염,"7개월 된 토이푸들 암컷을 보호하고 있습니다. 최근 몇 차례의 혼나는 상황 이후로,...",보호하고 계신 7개월 된 토이푸들은 이제 성장하여 매우 다양한 행동을 보일 시점에 ...
2,15,자견,내과,중독,반려견이 초콜릿을 섭취한 후에 이상한 증상을 보이고 있습니다. 학원에 가기 전 테가...,"문의해 주신 내용에 대해 자세히 검토하였습니다. 반려견이 초콜릿을 섭취할 경우, 이..."


In [14]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

db = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

collection = db._collection
print("collection count:", collection.count())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2754.68it/s]


collection count: 19206


## 답변 생성 프롬프트

검색된 Q&A 근거만 사용해 답변을 생성합니다. 정답 답변은 생성 단계에 넣지 않고, 생성 후 유사도 계산에만 사용합니다.

In [15]:
RAG_SYSTEM_PROMPT = """아래 [검색 근거]만 바탕으로 사용자의 반려견 건강 질문에 답하세요.
검색 근거에 없는 내용은 추측하지 말고, 확인이 필요한 부분은 병원 상담이 필요하다고 말하세요.
답변은 한국어로 간결하고 정확하게 작성하세요.

[검색 근거]
{context}"""

RAG_EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "{question}"),
])


def format_context(retrieved_docs) -> str:
    return "\n\n".join(
        f"질문: {doc.page_content}\n답변: {doc.metadata.get('qa.output', '')}"
        for doc, _score in retrieved_docs[:ANSWER_TOP_K]
    )


if OPENAI_API_KEY:
    generation_chain = (
        RAG_EVAL_PROMPT
        | ChatOpenAI(model=GENERATION_MODEL_NAME, temperature=0, api_key=OPENAI_API_KEY)
        | StrOutputParser()
    )
else:
    generation_chain = None


## 유사도와 검색 참고 지표 함수

In [16]:
def cosine_similarity(text_a: str, text_b: str) -> float:
    if not text_a or not text_b:
        return 0.0
    emb_a, emb_b = embedding_model.embed_documents([text_a, text_b])
    vec_a = np.array(emb_a, dtype=float)
    vec_b = np.array(emb_b, dtype=float)
    denominator = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denominator == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denominator)


def metadata_signature(metadata: dict) -> tuple:
    return tuple(metadata.get(key) for key in METADATA_KEYS)


def row_signature(row: pd.Series) -> tuple:
    return tuple(row[key] for key in METADATA_KEYS)


def is_relevant(metadata: dict, row: pd.Series) -> bool:
    return metadata_signature(metadata) == row_signature(row)


def precision_at_k(relevance_flags: list[bool], k: int) -> float:
    return sum(relevance_flags[:k]) / k if k else 0.0


def recall_at_k(relevance_flags: list[bool], total_relevant: int, k: int) -> float:
    if total_relevant <= 0:
        return 0.0
    return sum(relevance_flags[:k]) / total_relevant


def mrr_at_k(relevance_flags: list[bool], k: int) -> float:
    for rank, is_hit in enumerate(relevance_flags[:k], start=1):
        if is_hit:
            return 1 / rank
    return 0.0

In [17]:
def load_metadata_counts(collection, batch_size: int = 5000) -> Counter:
    counts = Counter()
    total = collection.count()
    for offset in range(0, total, batch_size):
        batch = collection.get(
            limit=batch_size,
            offset=offset,
            include=["metadatas"],
        )
        for metadata in batch.get("metadatas", []):
            counts[metadata_signature(metadata)] += 1
    return counts


metadata_counts = load_metadata_counts(collection)
print("metadata combinations:", len(metadata_counts))

metadata combinations: 247


## 단일 질문 평가

`retrieved_answer_similarity`는 OpenAI 호출 없이 검색된 근거 답변과 정답 답변의 최대 유사도를 봅니다.

`generated_answer_similarity`는 실제 RAG 생성 답변과 정답 답변의 유사도입니다.

In [18]:
def evaluate_one(row: pd.Series) -> dict:
    question = row["qa.input"]
    gold_answer = row["qa.output"]

    retrieved = db.similarity_search_with_score(question, k=MAX_K)
    retrieved_metadatas = [doc.metadata for doc, _score in retrieved]
    retrieved_scores = [_score for _doc, _score in retrieved]
    relevance_flags = [is_relevant(metadata, row) for metadata in retrieved_metadatas]
    total_relevant = metadata_counts.get(row_signature(row), 0)

    retrieved_answers = [doc.metadata.get("qa.output", "") for doc, _score in retrieved[:ANSWER_TOP_K]]
    retrieved_answer_similarities = [cosine_similarity(answer, gold_answer) for answer in retrieved_answers]
    retrieved_answer_similarity = max(retrieved_answer_similarities, default=0.0)

    context = format_context(retrieved)
    if generation_chain is None:
        generated_answer = ""
        generated_answer_similarity = None
    else:
        generated_answer = generation_chain.invoke({"context": context, "question": question})
        generated_answer_similarity = cosine_similarity(generated_answer, gold_answer)

    result = {
        "row_index": int(row.name),
        "question": question,
        "gold_answer": gold_answer,
        "generated_answer": generated_answer,
        "generated_answer_similarity": generated_answer_similarity,
        "retrieved_answer_similarity": retrieved_answer_similarity,
        "gold_lifeCycle": row["meta.lifeCycle"],
        "gold_department": row["meta.department"],
        "gold_disease": row["meta.disease"],
        "total_relevant": total_relevant,
        "top1_distance": retrieved_scores[0] if retrieved_scores else None,
        "top1_lifeCycle": retrieved_metadatas[0].get("meta.lifeCycle") if retrieved_metadatas else None,
        "top1_department": retrieved_metadatas[0].get("meta.department") if retrieved_metadatas else None,
        "top1_disease": retrieved_metadatas[0].get("meta.disease") if retrieved_metadatas else None,
    }

    for k in K_VALUES:
        result[f"precision@{k}"] = precision_at_k(relevance_flags, k)
        result[f"recall@{k}"] = recall_at_k(relevance_flags, total_relevant, k)
        result[f"mrr@{k}"] = mrr_at_k(relevance_flags, k)
        result[f"hit@{k}"] = int(any(relevance_flags[:k]))

    return result

## 전체 평가 실행

OpenAI API 키가 없으면 `generated_answer_similarity`는 계산되지 않고, 검색 근거 답변 기반 `retrieved_answer_similarity`만 계산됩니다.

In [19]:
results = []

for idx, row in eval_df.iterrows():
    results.append(evaluate_one(row))
    if (idx + 1) % 10 == 0 or idx + 1 == len(eval_df):
        print(f"evaluated {idx + 1}/{len(eval_df)}")

result_df = pd.DataFrame(results)
display(result_df.head())

evaluated 10/50
evaluated 20/50
evaluated 30/50
evaluated 40/50
evaluated 50/50


,row_index,question,gold_answer,generated_answer,generated_answer_similarity,retrieved_answer_similarity,gold_lifeCycle,gold_department,gold_disease,total_relevant,...,mrr@1,hit@1,precision@3,recall@3,mrr@3,hit@3,precision@5,recall@5,mrr@5,hit@5
0,0,저희 집에서 기르고 있는 강아지는 11세의 수컷입니다. 얼마 전 동물병원에 방문했을...,답변드립니다. 1. 청진을 통해 심장에 잡음이 발생하는 것을 확인할 수 있었습니다....,1. **심장에 기생충이 있을 수 있나요?** \n가능성은 있습니다. 심장사상충은...,0.716961,0.675609,노령견,내과,심장사상충,111,...,0.0,0,0.000000,0.000000,0.000000,0,0.2,0.009009,0.200000,1
1,1,"7개월 된 토이푸들 암컷을 보호하고 있습니다. 최근 몇 차례의 혼나는 상황 이후로,...",보호하고 계신 7개월 된 토이푸들은 이제 성장하여 매우 다양한 행동을 보일 시점에 ...,"요실금에서도 비슷한 증상이 나타날 수 있지만, **7개월령이라 요실금 가능성은 낮은...",0.745132,0.702431,성견,내과,방광염,40,...,1.0,1,0.333333,0.025000,1.000000,1,0.2,0.025000,1.000000,1
2,2,반려견이 초콜릿을 섭취한 후에 이상한 증상을 보이고 있습니다. 학원에 가기 전 테가...,"문의해 주신 내용에 대해 자세히 검토하였습니다. 반려견이 초콜릿을 섭취할 경우, 이...",현재 초콜릿 섭취 후 계속 켁켁거리고 이상한 소리를 낸다면 **상태를 지켜보지 말고...,0.841028,0.893398,자견,내과,중독,128,...,0.0,0,0.333333,0.007812,0.333333,1,0.4,0.015625,0.333333,1
3,3,저희 집 5개월된 비글 강아지가 우연히 포도를 섭취하여 걱정이 되어 이와 관련하여 ...,"우선, 강아지의 품종이 비글이며, 현재 5개월 정도의 나이를 가진 것으로 보아 체중...","포도는 강아지에게 해로울 수 있으며, **신장 손상이나 급성 신부전증**처럼 생명을...",0.824834,0.800572,성견,내과,중독,136,...,0.0,0,0.000000,0.000000,0.000000,0,0.0,0.000000,0.000000,0
4,4,저의 집에서 기르고 있는 것으로 리트리버 강아지가 메로나를 매우 원하며 짖은 모습을...,강아지가 아가 스크림 막대까지 삼키게 된 점에 유의해 주셔야 합니다. 상황이 상당히...,"나무는 소화가 잘 되지 않습니다. 삼킨 조각이 작으면 배변으로 나올 수도 있지만, ...",0.823352,0.904140,성견,내과,위장관폐색,59,...,0.0,0,0.000000,0.000000,0.000000,0,0.0,0.000000,0.000000,0


## 답변 유사도 요약

In [20]:
similarity_columns = ["retrieved_answer_similarity"]
if "generated_answer_similarity" in result_df.columns and result_df["generated_answer_similarity"].notna().any():
    similarity_columns.insert(0, "generated_answer_similarity")

summary_rows = []
for column in similarity_columns:
    values = result_df[column].dropna()
    summary_rows.append({
        "metric": column,
        "count": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "min": values.min(),
        "max": values.max(),
        "ratio_ge_0.70": (values >= 0.70).mean(),
        "ratio_ge_0.80": (values >= 0.80).mean(),
        "ratio_ge_0.90": (values >= 0.90).mean(),
    })

answer_similarity_summary_df = pd.DataFrame(summary_rows)
display(answer_similarity_summary_df)

,metric,count,mean,median,min,max,ratio_ge_0.70,ratio_ge_0.80,ratio_ge_0.90
0,generated_answer_similarity,50,0.744791,0.754690,0.420792,0.908454,0.7,0.34,0.02
1,retrieved_answer_similarity,50,0.772107,0.786476,0.509359,0.962862,0.8,0.42,0.08


## 검색 지표 참고 요약

In [21]:
retrieval_summary_rows = []

for k in K_VALUES:
    retrieval_summary_rows.append({
        "k": k,
        "precision": result_df[f"precision@{k}"].mean(),
        "recall": result_df[f"recall@{k}"].mean(),
        "mrr": result_df[f"mrr@{k}"].mean(),
        "hit_rate": result_df[f"hit@{k}"].mean(),
    })

retrieval_summary_df = pd.DataFrame(retrieval_summary_rows)
display(retrieval_summary_df)

,k,precision,recall,mrr,hit_rate
0,1,0.040000,0.005500,0.040000,0.04
1,3,0.073333,0.007940,0.106667,0.20
2,5,0.068000,0.008973,0.115667,0.24


## 유사도 낮은 케이스 확인

In [22]:
sort_column = "generated_answer_similarity" if result_df["generated_answer_similarity"].notna().any() else "retrieved_answer_similarity"

low_similarity_columns = [
    "row_index",
    sort_column,
    "retrieved_answer_similarity",
    "gold_lifeCycle",
    "gold_department",
    "gold_disease",
    "top1_lifeCycle",
    "top1_department",
    "top1_disease",
    "question",
    "gold_answer",
    "generated_answer",
]

low_similarity = result_df.sort_values(sort_column, ascending=True)
display(low_similarity[low_similarity_columns].head(20))

,row_index,generated_answer_similarity,retrieved_answer_similarity,gold_lifeCycle,gold_department,gold_disease,top1_lifeCycle,top1_department,top1_disease,question,gold_answer,generated_answer
17,17,0.420792,0.798567,노령견,내과,심장사상충,자견,내과,기타,"저희가 어제 유기견을 입양하였는데, 그 어린 강아지가 지속적으로 켁켁거리며 가끔 구...","보호자님, 유기견 입양을 축하드립니다. 어린 강아지가 지속적으로 켁켁거리고 침만 나...",지속적인 **켁켁거림과 침만 나오는 구토 같은 행동**만으로는 원인을 확정하기 어렵...
33,33,0.435257,0.509359,성견,내과,심장사상충,성견,피부과,기타,"사상충 치료를 받고 있는 강아지를 치료가 완료될 때까지 임시 보호하고자 하는데, 현...",직접적인 접촉을 피하신다면 큰 문제는 없을 것으로 생각됩니다. 평소에 심장사상충 예...,제시된 근거만으로는 **사상충 치료 중인 강아지가 다른 반려견에게 전파될 수 있는지...
44,44,0.548900,0.553705,노령견,내과,심장사상충,성견,내과,심장사상충,"강아지에게 레볼루션을 바른 후, 그것이 제 손에 묻어버리는 상황이 발생하였습니다. ...","만약 사람의 피부에 약물이 묻었다면, 즉시 해당 부위를 깨끗이 씻어내고, 눈이나 입...",제공된 근거에는 **레볼루션이 사람 피부에 묻었을 때의 안전성이나 대처 방법이 나와...
21,21,0.581769,0.747676,노령견,내과,중독,성견,내과,기타,안녕하세요. 저희 집에 있는 강아지는 6개월 된 진돗개입니다. 사료는 하루에 두 번...,"귀하의 강아지의 상태를 고려할 때, 가능한 한 빨리 내륙에 위치한 동물병원으로 가시...",현재 상태는 **응급하게 재진·정밀 진료가 필요한 상황**입니다. 특히 다음이 우려...
31,31,0.629577,0.763316,노령견,내과,방광염,자견,내과,기타,"1살 된 강아지가 요로결석을 유발하는 세균에 감염되어 영양제와 사료, 약물까지 처방...",강아지가 1살이 라는 점에서 매우 걱정되는 상황임을 이해합니다. 일반적으로 요로결석...,소변에 피가 **점점 많이 섞이는 것은 호전되지 않거나 악화되는 신호일 수 있으므로...
12,12,0.644287,0.848089,노령견,내과,무릎뼈 탈구,성견,외과,무릎뼈 탈구,"강아지가 갑자기 쓰러졌습니다. 오후 2시부터 11살 된 말티즈가 잠을 자고 있었고,...","강아지가 겪고 있는 것으로 문제는 신경계, 정형외와, 소화기와 관련된 다양한 요인들...",갑자기 비틀거리며 쓰러지고 다리에 힘이 빠진 증상은 일시적으로 회복되었더라도 정상으...
13,13,0.660991,0.785377,노령견,내과,무릎뼈 탈구,자견,내과,기타,강아지가 신부전과 열사병 진단을 받은 1년 반 된 남성 말티즈를 소유하고 있습니다....,강아지의 상태와 관련하여 여러 가지 요인들을 고려할 필요가 있습니다. 여름철에는 열...,현재 증상만으로 정확한 원인을 단정할 수 없습니다. 다만 검색 근거상 다음 사항은 ...
16,16,0.661005,0.627904,노령견,내과,심장사상충,자견,외과,기타,강아지의 성대 제거 수술 후에 제공되는 약물과 심장사상충 예방약을 함께 복용해도 괜...,"네, 그 점에 관해서는 문제가 없습니다. 그러나 심장사상충 예방약은 약간의 시간 간...",성대 제거 수술 후 처방받은 약물과 심장사상충 예방약을 함께 복용해도 되는지는 약의...
43,43,0.673762,0.665122,성견,내과,심장사상충,노령견,내과,심장사상충,집에 대형견이 심장사상충에 감염되었다고 합니다. 저희 대형견이 오늘 동물병원에서 진...,심장사상충은 주로 모기에 의해 전파되는 질병입니다. 감염된 유충이 강아지의 체내에 ...,"심장사상충은 상태에 따라 치료 계획이 달라지므로, 진단받은 동물병원에서 **감염 단..."
32,32,0.675730,0.669867,성견,내과,중독,성견,내과,기타,제 강아지가 양파를 섭취한 것 같습니다. 강아지가 엄지손가락 크기의 양파를 1-2개...,"귀하의 강아지를 동물병원으로 데려가면, 소심한 성격의 아이가 더욱 예민해져 진료 과...","걱정되시겠지만, **양파는 강아지의 적혈구를 파괴해 용혈성 빈혈을 일으킬 수 있는 ..."


## 질병별 답변 유사도

In [23]:
agg_dict = {
    "count": ("row_index", "count"),
    "retrieved_answer_similarity": ("retrieved_answer_similarity", "mean"),
    "hit_at_3": ("hit@3", "mean"),
}

if result_df["generated_answer_similarity"].notna().any():
    agg_dict["generated_answer_similarity"] = ("generated_answer_similarity", "mean")

by_disease_df = (
    result_df
    .groupby("gold_disease")
    .agg(**agg_dict)
    .sort_values("retrieved_answer_similarity", ascending=True)
)

display(by_disease_df.head(20))

,count,retrieved_answer_similarity,hit_at_3,generated_answer_similarity
gold_disease,,,,
중성화수술,1,0.694784,0.000,0.700274
방광결석,2,0.726043,0.000,0.739122
부신피질기능항진증,2,0.734958,0.500,0.717189
방광염,5,0.741580,0.400,0.729699
심장사상충,16,0.753209,0.125,0.737365
중독,16,0.795227,0.250,0.769467
위장관폐색,4,0.805036,0.000,0.779337
무릎뼈 탈구,2,0.816733,0.000,0.652639
단두종증후군,1,0.820866,1.000,0.760947


## 결과 저장

In [24]:
OUTPUT_DIR = PROJECT_DIR / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

result_path = OUTPUT_DIR / "rag_answer_similarity_eval_results.csv"
answer_summary_path = OUTPUT_DIR / "rag_answer_similarity_eval_summary.csv"
retrieval_summary_path = OUTPUT_DIR / "rag_retrieval_reference_summary.csv"
by_disease_path = OUTPUT_DIR / "rag_answer_similarity_by_disease.csv"

result_df.to_csv(result_path, index=False, encoding="utf-8-sig")
answer_similarity_summary_df.to_csv(answer_summary_path, index=False, encoding="utf-8-sig")
retrieval_summary_df.to_csv(retrieval_summary_path, index=False, encoding="utf-8-sig")
by_disease_df.to_csv(by_disease_path, encoding="utf-8-sig")

print("saved:")
print(result_path)
print(answer_summary_path)
print(retrieval_summary_path)
print(by_disease_path)

saved:
c:\Users\rhksa\Desktop\mle-01-p1-team2\notebooks\outputs\rag_answer_similarity_eval_results.csv
c:\Users\rhksa\Desktop\mle-01-p1-team2\notebooks\outputs\rag_answer_similarity_eval_summary.csv
c:\Users\rhksa\Desktop\mle-01-p1-team2\notebooks\outputs\rag_retrieval_reference_summary.csv
c:\Users\rhksa\Desktop\mle-01-p1-team2\notebooks\outputs\rag_answer_similarity_by_disease.csv
